# AI Text Detector — Train 4-Class Classifier

Fine-tunes DeBERTa-v3-base for 4-class AI text detection:
- 0: Pure Human
- 1: Pure AI (27 models × 7 prompt styles)
- 2: AI + Polished (AI text rewritten by another model)
- 3: Human + Polished (human text polished by AI)

**Instructions:**
1. Upload `dataset.jsonl` to Google Drive at `MyDrive/ai-text-detector/`
2. Runtime → Change runtime type → **T4 GPU**
3. Runtime → Run all (~2-3 hours)
4. Model auto-saves to Google Drive

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn seaborn

In [ ]:
# Mount Google Drive — dataset.jsonl should be at MyDrive/ai-text-detector/dataset.jsonl
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ai-text-detector'
DATASET_FILE = os.path.join(DRIVE_DIR, 'dataset.jsonl')

if not os.path.exists(DATASET_FILE):
    # Fallback: upload manually
    print(f'{DATASET_FILE} not found. Upload manually:')
    from google.colab import files
    uploaded = files.upload()
    DATASET_FILE = list(uploaded.keys())[0]
else:
    size_mb = os.path.getsize(DATASET_FILE) / (1024*1024)
    print(f'Found dataset: {DATASET_FILE} ({size_mb:.0f} MB)')

In [ ]:
import json
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

LABEL_NAMES = ['human', 'ai', 'ai_polished', 'human_polished']
NUM_LABELS = 4
MODEL_NAME = 'microsoft/deberta-v3-base'

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
print(f'GPU: {gpu_name}')
print(f'CUDA: {torch.version.cuda}' if torch.cuda.is_available() else 'No CUDA')

In [ ]:
# Load dataset — only keep 'text' and 'label' fields
data = []
with open(DATASET_FILE, 'r') as f:
    for line in f:
        entry = json.loads(line)
        data.append({'text': entry['text'], 'label': entry['label']})

print(f'Total: {len(data)}')

# Integrity check
labels_present = set(d['label'] for d in data)
assert labels_present == {0, 1, 2, 3}, f'Missing labels: expected {{0,1,2,3}}, got {labels_present}'
assert all(0 <= d['label'] <= 3 for d in data), 'Label out of range'
for i, name in enumerate(LABEL_NAMES):
    count = sum(1 for d in data if d['label'] == i)
    assert count > 0, f'No samples for class {name}'
    print(f'  {name}: {count}')
print('Integrity check passed.')

# Deduplicate by text
seen = set()
deduped = []
for d in data:
    if d['text'] not in seen:
        seen.add(d['text'])
        deduped.append(d)
print(f'\nAfter dedup: {len(deduped)} ({len(data) - len(deduped)} duplicates removed)')
data = deduped

# Balance classes: downsample to smallest class
from collections import Counter
class_counts = Counter(d['label'] for d in data)
min_count = min(class_counts.values())
print(f'Balancing to {min_count} per class...')

import random
random.seed(42)
balanced = []
for label in range(NUM_LABELS):
    class_data = [d for d in data if d['label'] == label]
    balanced.extend(random.sample(class_data, min_count))
random.shuffle(balanced)
data = balanced
print(f'Balanced: {len(data)} total ({min_count} × {NUM_LABELS})')

# Split 80/10/10 — val for early stopping, test for final evaluation
from sklearn.model_selection import train_test_split
train_data, temp_data = train_test_split(
    data, test_size=0.2, random_state=42,
    stratify=[d['label'] for d in data],
)
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42,
    stratify=[d['label'] for d in temp_data],
)

train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)
test_ds = Dataset.from_list(test_data)

print(f'\nTrain: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')
print('Label distribution (train):')
for i, name in enumerate(LABEL_NAMES):
    count = sum(1 for d in train_data if d['label'] == i)
    print(f'  {name}: {count}')

In [ ]:
# Tokenize — dynamic padding (no max_length padding, done per-batch by collator)
from transformers import DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512)

# Don't set_format('torch') — DataCollatorWithPadding needs Python lists
# to dynamically pad each batch to its longest sequence
train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=['text'])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=['text'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f'Tokenized with dynamic padding. Features: {train_ds.column_names}')
print(f'Sample lengths: {[len(train_ds[i]["input_ids"]) for i in range(5)]}')

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={i: n for i, n in enumerate(LABEL_NAMES)},
    label2id={n: i for i, n in enumerate(LABEL_NAMES)},
)

# DeBERTa-v3 has internal fp16 embedding layers that cause gradient issues.
# Force all parameters to fp32 before training. This is necessary even with
# fp16=False in TrainingArguments, because the model weights themselves are fp16.
model = model.float()

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {total_params:,} total, {trainable:,} trainable')

# Verify all params are fp32
dtypes = set(str(p.dtype) for p in model.parameters())
print(f'Parameter dtypes: {dtypes}')

In [ ]:
# Training config — auto-detects GPU capabilities
import shutil, subprocess
from safetensors.torch import load_file as safe_load_file

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

OUTPUT_DIR = os.path.join(DRIVE_DIR, 'detector_checkpoints')
FINAL_DIR = os.path.join(DRIVE_DIR, 'detector_model')

# Clean old checkpoints
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
    print(f'Cleaned old checkpoints: {OUTPUT_DIR}')

# Detect GPU tier by name + VRAM
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
is_high_end = any(x in gpu_name for x in ['A100', 'L4', 'H100', 'A10'])
use_bf16 = is_high_end

# batch_size based on actual VRAM
# A100-80GB=64, A100-40GB/L4=32, T4-16GB=16
if gpu_mem_gb > 60:
    batch_size = 64
elif gpu_mem_gb > 30:
    batch_size = 32
else:
    batch_size = 16
grad_accum = max(1, 64 // batch_size)

print(f'GPU: {gpu_name} ({gpu_mem_gb:.0f} GB) → {"bf16" if use_bf16 else "fp32"}, batch={batch_size}, grad_accum={grad_accum}')

pre_std = model.classifier.weight.data.std().item()
print(f'Pre-training classifier std: {pre_std:.4f}')

NUM_EPOCHS = 4  # Full training

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    seed=42,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 2,
    gradient_accumulation_steps=grad_accum,
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='no',
    load_best_model_at_end=False,
    logging_steps=50,
    fp16=False,
    bf16=use_bf16,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

eff_batch = batch_size * grad_accum
steps = len(train_ds) // eff_batch * NUM_EPOCHS
print(f'\nTraining: {NUM_EPOCHS} epochs, ~{steps} steps (eff_batch={eff_batch})\n')
train_result = trainer.train()

# ═══════════════════════════════════════════════════════════════
# AUTO-SAVE — immediately after training
# ═══════════════════════════════════════════════════════════════
post_std = model.classifier.weight.data.std().item()
print(f'\n{"="*60}')
print(f'Loss: {train_result.training_loss:.4f}')
print(f'Classifier std: {pre_std:.4f} → {post_std:.4f}')
print(f'Status: training complete\n')

# Save
os.makedirs(FINAL_DIR, exist_ok=True)
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

# Verify
saved_w = safe_load_file(os.path.join(FINAL_DIR, 'model.safetensors'))
cls_key = [k for k in saved_w if 'classifier.weight' in k][0]
saved_std = saved_w[cls_key].std().item()
print(f'Saved std: {saved_std:.4f}')

# Zip
zip_path = os.path.join(DRIVE_DIR, 'detector_model.zip')
if os.path.exists(zip_path):
    os.remove(zip_path)
subprocess.run(['zip', '-r', zip_path, 'detector_model/'], cwd=DRIVE_DIR, check=True)
size = os.path.getsize(zip_path) / (1024*1024)
print(f'✓ detector_model.zip ({size:.0f} MB)')
print(f'{"="*60}')

# Quick sanity test
print('\n=== Quick sanity test ===')
model.eval()
test_texts = [
    ("Human casual", "my cat knocked over the coffee this morning and I just stared at the mess for like 5 minutes"),
    ("AI style", "The rapid advancement of technology has fundamentally transformed the landscape of modern education."),
    ("Human news", "Scientists said many of the species relied on snow cover remaining high on hills until late spring."),
    ("Tweet", "just found out my roommate has been using my netflix for 6 months without telling me"),
]
for desc, text in test_texts:
    inputs = tokenizer(text, truncation=True, max_length=512, return_tensors='pt').to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1).cpu().numpy()[0]
    pred = LABEL_NAMES[int(probs.argmax())]
    print(f'  [{desc}] → {pred} | h:{probs[0]*100:.0f}% ai:{probs[1]*100:.0f}% ap:{probs[2]*100:.0f}% hp:{probs[3]*100:.0f}%')

In [ ]:
# Training summary + loss curve
print('=== Training Summary ===')
print(f'  Total steps: {train_result.global_step}')
print(f'  Train loss: {train_result.training_loss:.4f}')
for k, v in train_result.metrics.items():
    print(f'  {k}: {v}')

# Plot loss curve from trainer log history
log_history = trainer.state.log_history
train_steps = [h['step'] for h in log_history if 'loss' in h]
train_losses = [h['loss'] for h in log_history if 'loss' in h]
eval_steps = [h['step'] for h in log_history if 'eval_loss' in h]
eval_losses = [h['eval_loss'] for h in log_history if 'eval_loss' in h]
eval_accs = [h['eval_accuracy'] for h in log_history if 'eval_accuracy' in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_steps, train_losses, label='Train', alpha=0.7)
ax1.plot(eval_steps, eval_losses, label='Val', marker='o', markersize=4)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(eval_steps, eval_accs, marker='o', color='green', markersize=4)
ax2.set_xlabel('Step')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'training_curves.png'), dpi=150)
plt.show()
print(f'Saved: {DRIVE_DIR}/training_curves.png')

In [ ]:
# Final evaluation on held-out TEST set (not used during training or early stopping)
preds_output = trainer.predict(test_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

print('=== Test Set Classification Report ===')
print(classification_report(labels, preds, target_names=LABEL_NAMES, digits=3))
print(f'Overall accuracy: {accuracy_score(labels, preds):.3f}')

# Confusion matrix
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

# Per-class accuracy
print('\nPer-class accuracy:')
for i, name in enumerate(LABEL_NAMES):
    mask = labels == i
    if mask.sum() > 0:
        acc = (preds[mask] == i).mean()
        print(f'  {name}: {acc:.3f} ({mask.sum()} samples)')
print(f'\nSaved: {DRIVE_DIR}/confusion_matrix.png')

In [ ]:
# Save final model to Google Drive + cleanup checkpoints
import shutil

FINAL_DIR = os.path.join(DRIVE_DIR, 'detector_model')
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

# Zip for easy download (remove old zip first)
zip_path = os.path.join(DRIVE_DIR, 'detector_model.zip')
if os.path.exists(zip_path):
    os.remove(zip_path)
!cd "{DRIVE_DIR}" && zip -r detector_model.zip detector_model/

size = os.path.getsize(zip_path) / (1024*1024)
print(f'Model saved to: {FINAL_DIR}/')
print(f'Zip: {zip_path} ({size:.0f} MB)')

# Cleanup checkpoint dir to free Drive space
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
    print(f'Cleaned up checkpoints: {OUTPUT_DIR}')

print(f'\nAll outputs in {DRIVE_DIR}/:')
for f in sorted(os.listdir(DRIVE_DIR)):
    p = os.path.join(DRIVE_DIR, f)
    if os.path.isfile(p):
        s = os.path.getsize(p)
    elif os.path.isdir(p):
        s = sum(os.path.getsize(os.path.join(dp, fn))
                for dp, _, fns in os.walk(p) for fn in fns)
    else:
        s = 0
    print(f'  {f}: {s/(1024*1024):.0f} MB')

## After Training

Model and diagnostics are saved to Google Drive at `MyDrive/ai-text-detector/`:
- `detector_model/` — model weights + tokenizer
- `detector_model.zip` — zipped for download
- `training_curves.png` — loss and accuracy curves
- `confusion_matrix.png` — test set confusion matrix

To use locally:
1. Download `detector_model.zip` from Drive
2. Unzip into `ai-text-detector/models/detector/`
3. Update the Python server to load this model for classification